In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!rm -rf /kaggle/working/*
!pip install -q sentence-transformers

In [ ]:
# ── Cell 2: Paths + config ────────────────────────────────────────────────────
import os, json

KAGGLE_INPUT_DIR     = '/kaggle/input/'
KAGGLE_OUTPUT_DIR    = '/kaggle/working'
KAGGLE_USERNAME      = 'drunktolstoy'
KAGGLE_OUTPUT_DATASET = 'knesset-rag-idx'

os.chdir(KAGGLE_OUTPUT_DIR)
print('Working directory:', os.getcwd())

DATA_GLOB    = os.path.join(KAGGLE_INPUT_DIR, '*/**/*.json')
INDEX_PATH   = os.path.join(KAGGLE_OUTPUT_DIR, 'rag_index.npz')
CHUNK_SIZE   = 20   # utterances per chunk — covers a typical topic shift
CHUNK_STRIDE = 10   # 50% overlap — ensures topics aren't split at boundaries
EMBED_MODEL  = 'intfloat/multilingual-e5-small'
BATCH_SIZE   = 512

In [ ]:
# ── Cell 3: Build index (multilingual-e5-small) ───────────────────────────────
import glob as _glob, re, numpy as np, subprocess
from sentence_transformers import SentenceTransformer

CHECKPOINT_PATH  = os.path.join(KAGGLE_OUTPUT_DIR, 'rag_index_checkpoint.npz')
CHECKPOINT_EVERY = 200

MARKER_RE = re.compile(r'<<[^>]*>>')

def clean(text):
    return MARKER_RE.sub('', text).strip()

def load_session(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def make_chunks(utterances, size=CHUNK_SIZE, stride=CHUNK_STRIDE):
    return [utterances[i: i + size]
            for i in range(0, len(utterances), stride)
            if utterances[i: i + size]]

def _ensure_dataset_metadata():
    meta_path = os.path.join(KAGGLE_OUTPUT_DIR, 'dataset-metadata.json')
    if not os.path.exists(meta_path):
        with open(meta_path, 'w') as f:
            json.dump({
                'title': KAGGLE_OUTPUT_DATASET,
                'id': f'{KAGGLE_USERNAME}/{KAGGLE_OUTPUT_DATASET}',
                'licenses': [{'name': 'CC0-1.0'}],
            }, f)

def _dataset_exists():
    r = subprocess.run(
        ['kaggle', 'datasets', 'list', '--mine', '--search', KAGGLE_OUTPUT_DATASET],
        capture_output=True, text=True,
    )
    return KAGGLE_OUTPUT_DATASET in r.stdout

def push_to_kaggle_dataset(label='checkpoint'):
    _ensure_dataset_metadata()
    cmd = (
        ['kaggle', 'datasets', 'version', '-p', KAGGLE_OUTPUT_DIR, '-m', label, '--dir-mode', 'zip']
        if _dataset_exists()
        else ['kaggle', 'datasets', 'create', '-p', KAGGLE_OUTPUT_DIR, '--dir-mode', 'zip']
    )
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f'  [kaggle push] {label} → {KAGGLE_USERNAME}/{KAGGLE_OUTPUT_DATASET}')
    else:
        print(f'  [kaggle push FAILED]\n{result.stderr.strip()}')

def save_checkpoint(all_vecs, all_meta, processed_paths, push=True):
    np.savez_compressed(
        CHECKPOINT_PATH,
        vecs=np.stack(all_vecs),
        metadata=np.array(all_meta, dtype=object),
        processed_paths=np.array(list(processed_paths)),
    )
    print(f'  [checkpoint] {len(all_meta)} chunks, {len(processed_paths)} sessions → {CHECKPOINT_PATH}')
    if push:
        push_to_kaggle_dataset(label=f'checkpoint-{len(processed_paths)}-sessions')

def load_checkpoint():
    candidates = [
        CHECKPOINT_PATH,
        os.path.join('/kaggle/input', KAGGLE_OUTPUT_DATASET, 'rag_index_checkpoint.npz'),
    ]
    for path in candidates:
        if os.path.exists(path):
            data = np.load(path, allow_pickle=True)
            all_vecs   = list(data['vecs'])
            all_meta   = data['metadata'].tolist()
            done_paths = set(data['processed_paths'].tolist())
            print(f'  [checkpoint] resumed: {len(all_meta)} chunks, {len(done_paths)} sessions done')
            return all_vecs, all_meta, done_paths
    return [], [], set()

# ── build ─────────────────────────────────────────────────────────────────────

if os.path.exists(INDEX_PATH):
    print(f'Loading cached index from {INDEX_PATH} …')
    data     = np.load(INDEX_PATH, allow_pickle=True)
    vecs     = data['vecs']
    metadata = data['metadata'].tolist()
    print(f'Loaded {len(metadata)} chunks.')
else:
    print(f'Building index with {EMBED_MODEL} …')
    model = SentenceTransformer(EMBED_MODEL, device='cuda')

    all_vecs, all_meta, done_paths = load_checkpoint()
    paths     = sorted(_glob.glob(DATA_GLOB, recursive=True))
    remaining = [p for p in paths if p not in done_paths]
    print(f'Sessions total: {len(paths)}, remaining: {len(remaining)}')

    for p_idx, path in enumerate(remaining):
        if p_idx % 100 == 0:
            print(f'  {p_idx}/{len(remaining)} …')
        try:
            doc  = load_session(path)
            utts = doc.get('utterances', [])
            if not utts:
                done_paths.add(path)
                continue

            doc_id = os.path.splitext(os.path.basename(path))[0]
            title  = doc.get('title', '')
            date   = doc.get('date', doc.get('session_date', ''))[:10]
            url    = doc.get('url', doc.get('source_file', ''))

            # "passage: " prefix is required by multilingual-e5
            texts    = [f'passage: {clean(u.get("text", ""))}' for u in utts]
            utt_vecs = model.encode(
                texts,
                batch_size=BATCH_SIZE,
                normalize_embeddings=True,
                show_progress_bar=False,
                convert_to_numpy=True,
            )

            for i, chunk in enumerate(make_chunks(utts)):
                start     = i * CHUNK_STRIDE
                end       = start + len(chunk)
                chunk_vec = utt_vecs[start:end].mean(axis=0)
                norm = np.linalg.norm(chunk_vec)
                if norm > 1e-9:
                    chunk_vec /= norm

                all_vecs.append(chunk_vec.astype(np.float32))
                all_meta.append({
                    'doc_id': doc_id, 'title': title,
                    'date': date, 'url': url,
                    'from': int(chunk[0]['id']),
                    'to':   int(chunk[-1]['id']),
                    'utterances': chunk,
                })
            done_paths.add(path)
        except Exception as e:
            print(f'  [WARN] {path}: {e}')

        if (p_idx + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(all_vecs, all_meta, done_paths, push=True)

    vecs     = np.stack(all_vecs)
    metadata = all_meta
    np.savez_compressed(INDEX_PATH, vecs=vecs,
                        metadata=np.array(metadata, dtype=object))
    push_to_kaggle_dataset(label='final-index')
    if os.path.exists(CHECKPOINT_PATH):
        os.remove(CHECKPOINT_PATH)
    print(f'Done: {len(metadata)} chunks → {INDEX_PATH}')